[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davis-mironga/kitui-washlab-analysis/blob/main/notebooks/03_Hotspot_Analysis.ipynb)


# Notebook 03 — Hotspot Analysis
**Project:** WASHLAB Climate-Smart WASH Pilot — Kitui County  
**Analyst:** Davis Mironga  
**Purpose:** Identify statistically significant spatial clusters of high and low water access stress across Kitui County.

**Requires:** Outputs from Notebook 02 in `Kitui_WASHLAB/outputs/`

---

## Analytical approach

This notebook runs two complementary spatial statistics:

| Method | Scale | Question answered |
|--------|-------|-------------------|
| Moran's I | Ward (global) | Is water stress spatially clustered across the county? |
| Getis-Ord Gi* | Pixel (local) | Where exactly are the clusters located? |

Ward-level Gi* is not used here because 40 units is too few to achieve statistical significance with a local statistic. Running Gi* on the 500m raster grid (~28,000 valid pixels) gives sufficient sample size for reliable local cluster detection.

## Outputs
- `kitui_gistar_raster.tif` — Gi* z-score raster at 500m resolution
- `kitui_hotspot_classified.tif` — Classified hotspot/coldspot raster
- `kitui_hotspot_ward.geojson` — Ward-level hotspot summary for Streamlit app
- `kitui_hotspot_summary.csv` — Ward table for report
- `kitui_hotspot_map.png` — Publication-ready map

> **Phase 2 note:** Coverage gap analysis (which wards have hotspot stress AND insufficient borehole coverage) will be added in Notebook 04 once the borehole dataset is received.


### 1. Setup

Installs required libraries and mounts Google Drive.
All inputs are read from `Kitui_WASHLAB/outputs/` where Notebook 02 saved them.


In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
!pip install geopandas esda libpysal rasterio scipy matplotlib -q

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.transform import from_bounds
from rasterio.enums import Resampling
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from scipy.ndimage import uniform_filter
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

import libpysal
from libpysal.weights import Queen
import esda
from esda.moran import Moran

from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/Kitui_WASHLAB/'
OUT   = DRIVE + 'outputs/'
WGS84 = 'EPSG:4326'

print('Setup complete')
print('Run after Notebook 02 outputs are confirmed in Drive')


### 2. Load WASI Raster and Ward Boundaries

Loads the WASI composite raster and ward boundaries produced by Notebook 02.
The raster is the primary input for the Gi* analysis.
Ward boundaries are used for aggregation and mapping.


In [ ]:
# ── Load WASI raster and ward boundaries ──────────────────────────────────────

# Load WASI raster
with rasterio.open(OUT + 'kitui_wasi_500m.tif') as src:
    wasi      = src.read(1).astype(np.float32)
    transform = src.transform
    crs       = src.crs
    profile   = src.profile.copy()

# Replace nodata with nan
wasi = np.where(np.isnan(wasi), np.nan, wasi)

# Load ward boundaries and join WASI table
wards = gpd.read_file(OUT + 'kitui_wasi_ward.geojson')
wasi_table = pd.read_csv(OUT + 'kitui_wasi_ward_table.csv')
wards = wards.merge(
    wasi_table.drop(columns=['WASI_mean', 'Stress_Class'], errors='ignore'),
    on='Ward', how='left'
)
if wards.crs is None:
    wards = wards.set_crs(WGS84)

GRID_H, GRID_W = wasi.shape

print(f'WASI raster: {GRID_W} x {GRID_H} pixels at 500m')
print(f'Valid pixels: {(~np.isnan(wasi)).sum():,}')
print(f'WASI range: {np.nanmin(wasi):.3f} to {np.nanmax(wasi):.3f}')
print(f'Wards loaded: {len(wards)}')
print(f'Missing WASI: {wards["WASI_mean"].isna().sum()}')


### 3. Global Spatial Autocorrelation — Moran's I

Moran's I tests whether water stress is spatially clustered across the county as a whole.
A positive and significant I value means high-stress wards tend to be near other high-stress wards.
This is the prerequisite check before running local hotspot analysis.

Queen contiguity weights are used — wards sharing any boundary point are considered neighbours.
999 permutations are used to compute the simulated p-value.


In [ ]:
# ── Global spatial autocorrelation — Moran's I ────────────────────────────────

# Project to UTM for geometrically accurate contiguity
UTM_CRS = 'EPSG:32637'
wards_utm = wards.to_crs(UTM_CRS).reset_index(drop=True)

# Queen contiguity weights
w = Queen.from_dataframe(wards_utm, use_index=False)
w.transform = 'r'  # row-standardise

print(f'Spatial weights: {w.n} wards, {w.mean_neighbors:.1f} avg neighbours')
if w.islands:
    print(f'WARNING: {len(w.islands)} isolated wards — check boundary file')

# Fill NaN WASI with county mean for weights calculation
wasi_values = wards_utm['WASI_mean'].fillna(wards_utm['WASI_mean'].mean()).values

# Moran's I
mi = Moran(wasi_values, w, permutations=999)

print()
print('Moran\'s I — Global Spatial Autocorrelation')
print(f'  I statistic: {mi.I:.4f}')
print(f'  Z-score:     {mi.z_norm:.3f}')
print(f'  P-value:     {mi.p_sim:.4f} (999 permutations)')
print()
if mi.p_sim < 0.05:
    direction = 'positive (clustering)' if mi.I > 0 else 'negative (dispersion)'
    print(f'  Result: Significant {direction} spatial autocorrelation in water stress.')
    print(f'  High-stress wards tend to cluster spatially — local hotspot analysis is appropriate.')
else:
    print(f'  Result: No significant global spatial autocorrelation detected.')
    print(f'  Water stress is distributed randomly across wards — hotspot patterns may be weak.')


### 4. Local Hotspot Analysis — Getis-Ord Gi* on Raster

Computes Gi* z-scores for every valid pixel in the WASI raster using a 3x3 pixel window.
At 500m resolution a 3x3 window represents a 1.5km neighbourhood.

Gi* asks: is this pixel AND its immediate neighbours all high (or all low) relative to the county mean?
A high positive z-score means the pixel sits in a cluster of high stress.
A high negative z-score means it sits in a cluster of low stress.

Ward-level Gi* is not used because 40 wards is insufficient for local significance testing.
Running on the raster gives approximately 28,000 valid observations — sufficient statistical power.

Significance thresholds:
- z > 2.576: Hotspot at 99% confidence
- z > 1.960: Hotspot at 95% confidence
- z > 1.645: Hotspot at 90% confidence
- z < -1.645 to -2.576: equivalent coldspot thresholds


In [ ]:
# ── Getis-Ord Gi* on WASI raster ──────────────────────────────────────────────

def getis_ord_gi_star(arr, window=3):
    """
    Compute Gi* z-scores for a 2D raster array using a square moving window.
    NaN pixels are excluded from statistics.
    Returns z-score array (positive = hotspot, negative = coldspot).
    """
    valid = ~np.isnan(arr)
    n_total = valid.sum()

    # Global statistics across all valid pixels
    x_bar = np.nanmean(arr)
    s     = np.nanstd(arr)

    # Local sum and count using uniform filter
    arr_filled  = np.where(valid, arr, 0.0)
    local_sum   = uniform_filter(arr_filled,          size=window, mode='constant') * (window**2)
    local_count = uniform_filter(valid.astype(float), size=window, mode='constant') * (window**2)

    # Gi* formula
    numerator   = local_sum - x_bar * local_count
    denominator = s * np.sqrt(
        (n_total * local_count - local_count**2) / (n_total - 1)
    )

    gi_z = np.where(
        valid & (denominator > 0),
        numerator / denominator,
        np.nan
    )
    return gi_z

print('Computing Gi* z-scores on WASI raster (3x3 window = 1.5km neighbourhood)...')
gi_z = getis_ord_gi_star(wasi, window=3)

valid_z = gi_z[~np.isnan(gi_z)]
print(f'Gi* complete')
print(f'  Valid pixels:         {len(valid_z):,}')
print(f'  Z-score range:        {valid_z.min():.2f} to {valid_z.max():.2f}')
print(f'  Mean z-score:         {valid_z.mean():.2f}')
print()
print(f'  Hotspot 99%  (z > 2.576):  {(valid_z > 2.576).sum():,} pixels  ({(valid_z > 2.576).mean()*100:.1f}%)')
print(f'  Hotspot 95%  (z > 1.960):  {(valid_z > 1.960).sum():,} pixels  ({(valid_z > 1.960).mean()*100:.1f}%)')
print(f'  Hotspot 90%  (z > 1.645):  {(valid_z > 1.645).sum():,} pixels  ({(valid_z > 1.645).mean()*100:.1f}%)')
print(f'  Not significant:           {((valid_z >= -1.645) & (valid_z <= 1.645)).sum():,} pixels')
print(f'  Coldspot 90% (z < -1.645): {(valid_z < -1.645).sum():,} pixels  ({(valid_z < -1.645).mean()*100:.1f}%)')
print(f'  Coldspot 95% (z < -1.960): {(valid_z < -1.960).sum():,} pixels  ({(valid_z < -1.960).mean()*100:.1f}%)')
print(f'  Coldspot 99% (z < -2.576): {(valid_z < -2.576).sum():,} pixels  ({(valid_z < -2.576).mean()*100:.1f}%)')


### 5. Classify and Export Hotspot Rasters

Creates two output rasters:
- The raw Gi* z-score raster for continuous visualisation
- A classified raster with integer codes for each hotspot/coldspot category

Classification codes:

| Code | Class |
|------|-------|
| 3 | Hotspot 99% |
| 2 | Hotspot 95% |
| 1 | Hotspot 90% |
| 0 | Not significant |
| -1 | Coldspot 90% |
| -2 | Coldspot 95% |
| -3 | Coldspot 99% |


In [ ]:
# ── Classify and export hotspot rasters ───────────────────────────────────────

# Classification codes
classified = np.full(gi_z.shape, np.nan, dtype=np.float32)
classified = np.where(gi_z > 2.576,  3, classified)
classified = np.where((gi_z > 1.960) & (gi_z <= 2.576),  2, classified)
classified = np.where((gi_z > 1.645) & (gi_z <= 1.960),  1, classified)
classified = np.where((gi_z >= -1.645) & (gi_z <= 1.645), 0, classified)
classified = np.where((gi_z < -1.645) & (gi_z >= -1.960), -1, classified)
classified = np.where((gi_z < -1.960) & (gi_z >= -2.576), -2, classified)
classified = np.where(gi_z < -2.576, -3, classified)
classified = np.where(np.isnan(gi_z), np.nan, classified)

# Save Gi* z-score raster
gistar_path = OUT + 'kitui_gistar_raster.tif'
with rasterio.open(
    gistar_path, 'w', driver='GTiff',
    height=GRID_H, width=GRID_W,
    count=1, dtype='float32',
    crs=crs, transform=transform,
    nodata=float('nan'), compress='lzw'
) as dst:
    dst.write(gi_z.astype(np.float32), 1)
print(f'Gi* raster saved: {gistar_path}')

# Save classified raster
class_path = OUT + 'kitui_hotspot_classified.tif'
with rasterio.open(
    class_path, 'w', driver='GTiff',
    height=GRID_H, width=GRID_W,
    count=1, dtype='float32',
    crs=crs, transform=transform,
    nodata=float('nan'), compress='lzw'
) as dst:
    dst.write(classified.astype(np.float32), 1)
print(f'Classified raster saved: {class_path}')


### 6. Ward-Level Hotspot Summary

Summarises the pixel-level Gi* results to ward level by calculating
the percentage of each ward's pixels that fall into each hotspot class.
A ward is classified as a hotspot if more than 30% of its pixels are in
a hotspot category at 95% confidence or above.


In [ ]:
# ── Ward-level hotspot summary ────────────────────────────────────────────────
import rasterstats

# Zonal stats on classified raster
stats_gi = rasterstats.zonal_stats(
    wards, class_path,
    stats=['mean', 'count'],
    nodata=float('nan'),
    add_stats={
        'hotspot_pct': lambda x: float((x[~np.isnan(x)] >= 2).sum()) / max(len(x[~np.isnan(x)]), 1)
    }
)

wards['Gi_Mean']     = [s['mean'] for s in stats_gi]
wards['Hotspot_Pct'] = [s['hotspot_pct'] for s in stats_gi]

# Classify ward based on dominant pixel class
def classify_ward(gi_mean, hotspot_pct):
    if pd.isna(gi_mean): return 'No data'
    if hotspot_pct >= 0.30: return 'Hotspot'
    if gi_mean <= -1.0:     return 'Coldspot'
    return 'Not significant'

wards['Ward_Class'] = [
    classify_ward(m, p)
    for m, p in zip(wards['Gi_Mean'], wards['Hotspot_Pct'])
]

print('Ward hotspot summary:')
print(wards['Ward_Class'].value_counts().to_string())
print()
print('Hotspot wards (>=30% pixels at 95%+ confidence):')
hotspot_wards = wards[wards['Ward_Class'] == 'Hotspot'].sort_values('Hotspot_Pct', ascending=False)
if len(hotspot_wards) > 0:
    print(hotspot_wards[['Ward', 'WASI_mean', 'Gi_Mean', 'Hotspot_Pct']]
          .round(3)
          .to_string(index=False))
else:
    print('No wards exceed the 30% hotspot pixel threshold.')
    print('Top wards by mean Gi* z-score:')
    print(wards.nlargest(10, 'Gi_Mean')[['Ward', 'WASI_mean', 'Gi_Mean', 'Hotspot_Pct']]
          .round(3)
          .to_string(index=False))


### 7. Export Ward Outputs

Saves the ward-level hotspot summary as GeoJSON and CSV.
These files are used in the Streamlit app and in the Phase 1 report.


In [ ]:
# ── Export ward hotspot outputs ────────────────────────────────────────────────

# GeoJSON for Streamlit app
geojson_cols = ['Ward', 'WASI_mean', 'Gi_Mean', 'Hotspot_Pct', 'Ward_Class', 'geometry']
wards[geojson_cols].to_file(OUT + 'kitui_hotspot_ward.geojson', driver='GeoJSON')
print(f'GeoJSON saved: {OUT}kitui_hotspot_ward.geojson')

# CSV for report
csv_cols = ['Ward', 'WASI_mean', 'Gi_Mean', 'Hotspot_Pct', 'Ward_Class']
wards[csv_cols].sort_values('Gi_Mean', ascending=False).to_csv(
    OUT + 'kitui_hotspot_summary.csv', index=False
)
print(f'CSV saved: {OUT}kitui_hotspot_summary.csv')


### 8. Hotspot Maps

Produces two panels:
- Left: continuous Gi* z-score raster showing the full gradient of spatial clustering
- Right: classified ward-level hotspot map for reporting

Red areas are statistically significant clusters of high water stress.
Blue areas are clusters of low stress.
Grey areas show no significant spatial clustering.


In [ ]:
# ── Hotspot maps ──────────────────────────────────────────────────────────────

import matplotlib.colors as mcolors
from rasterio.transform import array_bounds

bounds = array_bounds(GRID_H, GRID_W, transform)
ext    = [bounds[0], bounds[2], bounds[1], bounds[3]]

fig, axes = plt.subplots(1, 2, figsize=(20, 14))

# Panel A: continuous Gi* z-score raster
ax = axes[0]
vmax = max(abs(np.nanmin(gi_z)), abs(np.nanmax(gi_z)))
im = ax.imshow(
    gi_z, cmap='RdBu_r', vmin=-vmax, vmax=vmax,
    extent=ext, origin='upper', aspect='equal'
)
wards.boundary.plot(ax=ax, color='black', linewidth=0.4)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Gi* Z-score')
ax.set_title('Gi* Z-score (continuous)\nPositive = high stress cluster | Negative = low stress cluster',
             fontsize=10)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

# Panel B: classified ward map
ax = axes[1]
WARD_COLOURS = {
    'Hotspot':         '#C00000',
    'Not significant': '#D9D9D9',
    'Coldspot':        '#2E75B6',
    'No data':         '#F0F0F0',
}
for cls, colour in WARD_COLOURS.items():
    subset = wards[wards['Ward_Class'] == cls]
    if len(subset) > 0:
        subset.plot(ax=ax, color=colour, linewidth=0.5, edgecolor='white')

# Label hotspot wards
for _, row in wards[wards['Ward_Class'] == 'Hotspot'].iterrows():
    c = row.geometry.centroid
    ax.annotate(row['Ward'], xy=(c.x, c.y), fontsize=6.5,
                ha='center', va='center', fontweight='bold', color='white',
                bbox=dict(boxstyle='round,pad=0.15', fc='#C00000', alpha=0.6, ec='none'))

legend_patches = [
    mpatches.Patch(color='#C00000', label='Hotspot (>=30% pixels at p<0.05)'),
    mpatches.Patch(color='#D9D9D9', label='Not significant'),
    mpatches.Patch(color='#2E75B6', label='Coldspot'),
]
ax.legend(handles=legend_patches, loc='lower left', fontsize=9)
ax.set_title('Water Stress Hotspot Classification by Ward\nGetis-Ord Gi* | 500m raster | 1.5km neighbourhood',
             fontsize=10)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

plt.suptitle('Spatial Water Stress Clusters — Kitui County, Kenya',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT + 'kitui_hotspot_map.png', dpi=200, bbox_inches='tight')
plt.show()
print('Hotspot map saved')


### 9. Summary Statistics

Produces the summary numbers used in the Phase 1 report.
Run this cell last to confirm the analysis is complete.


In [ ]:
# ── Summary statistics for report ─────────────────────────────────────────────

total      = len(wards)
n_hotspot  = (wards['Ward_Class'] == 'Hotspot').sum()
n_coldspot = (wards['Ward_Class'] == 'Coldspot').sum()
n_ns       = (wards['Ward_Class'] == 'Not significant').sum()

valid_z    = gi_z[~np.isnan(gi_z)]
pct_hs99   = (valid_z > 2.576).mean() * 100
pct_hs95   = (valid_z > 1.960).mean() * 100
pct_cs99   = (valid_z < -2.576).mean() * 100
pct_cs95   = (valid_z < -1.960).mean() * 100

print('HOTSPOT ANALYSIS SUMMARY')
print('='*55)
print(f'Moran\'s I:              {mi.I:.4f} (p={mi.p_sim:.4f})')
print()
print('Pixel-level Gi* (500m raster):')
print(f'  Hotspot 99% (z>2.576):  {pct_hs99:.1f}% of valid pixels')
print(f'  Hotspot 95% (z>1.960):  {pct_hs95:.1f}% of valid pixels')
print(f'  Coldspot 95% (z<-1.960): {pct_cs95:.1f}% of valid pixels')
print(f'  Coldspot 99% (z<-2.576): {pct_cs99:.1f}% of valid pixels')
print()
print('Ward-level classification:')
print(f'  Hotspot wards:         {n_hotspot} of {total}')
print(f'  Coldspot wards:        {n_coldspot} of {total}')
print(f'  Not significant:       {n_ns} of {total}')
print()
print('Outputs saved to Drive:')
print('  kitui_gistar_raster.tif')
print('  kitui_hotspot_classified.tif')
print('  kitui_hotspot_ward.geojson')
print('  kitui_hotspot_summary.csv')
print('  kitui_hotspot_map.png')
print()
print('Phase 2 note: Coverage gap analysis (hotspot wards vs borehole')
print('coverage) will be added in Notebook 04 once borehole data is received.')
print()
print('Notebook 03 complete. Next: Notebook 04 — Coverage Gap Analysis')
